# PINN — Damped Mass-Spring-Damper (MCK)

This notebook applies a PINN to a **second-order ODE** and demonstrates two
implementation details that are essential in practice:

1. **Second-order residual via chained autodiff** — differentiating the network output
   twice with respect to $t$
2. **Residual normalisation** — dividing the residual by $k$ to keep all loss terms $\mathcal{O}(1)$

**System:** $m\ddot{x} + c\dot{x} + kx = 0$, with $m=1\,\text{kg}$,
$c=4\,\text{N·s/m}$, $k=400\,\text{N/m}$ (underdamped).

**Key insight:** The plain NN memorises the 10 training points and collapses to zero
outside the training window. The PINN extrapolates the decaying oscillation correctly
because the ODE is enforced over the full $[0, 1\,\text{s}]$ domain.


In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
from systems import MassSpringDamper

SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device: {DEVICE}")

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
COLORS = {"True": "#2c3e50", "PINN": "#3498db", "NN": "#e74c3c"}

## Physical Parameters and System

We use the **physical** $m, c, k$ parameterisation directly, rather than the
equivalent harmonic form $\ddot{x} + 2\delta\dot{x} + \omega_0^2 x = 0$.

The `MassSpringDamper` class takes $\delta = c/(2m)$ and $\omega_0 = \sqrt{k/m}$
internally, but we expose $m, c, k$ to the PINN so the residual reads as the
physical equation of motion.

With $k = 400\,\text{N/m}$, the spring term $kx \approx 400x$ is ~400× larger than
typical displacements. The residual is normalised by $k$ to prevent this from
dominating the data loss.


In [ ]:
M, C, K  = 1.0, 4.0, 400.0          # kg,  N·s/m,  N/m
mck      = MassSpringDamper(delta=C / (2*M), omega0=(K/M)**0.5)

T_TRAIN = (0.0, 0.36)
T_EVAL  = (0.0, 1.0)
N_TRAIN, N_EVAL = 10, 500

t_train = np.linspace(*T_TRAIN, N_TRAIN)
x_train = mck.solution(t_train)
t_eval  = np.linspace(*T_EVAL,  N_EVAL)
x_true  = mck.solution(t_eval)

t_tr = torch.tensor(t_train[:, None], dtype=torch.float32, device=DEVICE)
x_tr = torch.tensor(x_train[:, None], dtype=torch.float32, device=DEVICE)
t_ev = torch.tensor(t_eval[:,  None], dtype=torch.float32, device=DEVICE)

# Visualise training data vs full trajectory
plt.figure(figsize=(10, 3))
plt.plot(t_eval, x_true, color=COLORS["True"], lw=1.5, ls="--", label="Ground truth")
plt.scatter(t_train, x_train, s=60, color=COLORS["True"], zorder=5, label="10 training obs")
plt.axvline(T_TRAIN[1], color="gray", ls=":", lw=1.2, label="train end")
plt.axvspan(T_TRAIN[1], T_EVAL[1], alpha=0.06, color="gray")
plt.xlabel("t (s)"); plt.ylabel("x(t)"); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## PINN Model

The PINN maps $t \mapsto x(t)$ using a standard feedforward network.
The physics residual is:

$$\tilde{\mathcal{R}}[x_\theta](t) = \frac{m\,\ddot{x}_\theta + c\,\dot{x}_\theta + k\,x_\theta}{k}$$

Normalising by $k$ brings the residual to $\mathcal{O}(1)$, making the physics loss
comparable in magnitude to the data loss from the first epoch — no warm-up phase needed.

$\dot{x}$ and $\ddot{x}$ are both obtained by calling `autograd.grad` twice
(**chained** or **nested** autodiff). `create_graph=True` on the first call is
essential: it keeps the computational graph alive so the second derivative can
be computed.

**Collocation points** are sampled randomly over $[0, T_{\text{eval}}]$ at each
training step, ensuring the ODE is enforced in the extrapolation region too.


In [ ]:
class PINN(nn.Module):
    def __init__(self, m, c, k, hidden_dim=64, n_layers=4, t_scale=1.0):
        super().__init__()
        self.m, self.c, self.k, self.t_scale = m, c, k, t_scale
        layers = [nn.Linear(1, hidden_dim), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        return self.net(t / self.t_scale)

    def residual(self, t_c):
        t_r = t_c.detach().requires_grad_(True)
        x   = self.forward(t_r)
        xd  = torch.autograd.grad(x.sum(),  t_r, create_graph=True)[0]
        xdd = torch.autograd.grad(xd.sum(), t_r, create_graph=True)[0]
        return (self.m * xdd + self.c * xd + self.k * x) / self.k   # O(1)

    def total_loss(self, t_obs, x_obs, t_c, w_data=1.0, w_phys=1.0):
        ld = ((self.forward(t_obs) - x_obs)**2).mean()
        lp = (self.residual(t_c)**2).mean()
        return w_data*ld + w_phys*lp, ld, lp


class TrajNN(nn.Module):
    def __init__(self, hidden_dim=64, n_layers=4, t_scale=1.0):
        super().__init__()
        self.t_scale = t_scale
        layers = [nn.Linear(1, hidden_dim), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        return self.net(t / self.t_scale)

## Training

The PINN is trained with data loss + physics loss from the first epoch.
Because the residual is normalised by $k$, both terms are $\mathcal{O}(1)$
at initialisation — no warm-up phase is required.

The plain NN is trained on data only for 3 000 epochs.


In [ ]:
EPOCHS_PINN = 20_000

pinn = PINN(M, C, K, t_scale=T_EVAL[1]).to(DEVICE)
opt  = torch.optim.Adam(pinn.parameters(), lr=1e-3)
sch  = torch.optim.lr_scheduler.MultiStepLR(
    opt, milestones=[int(EPOCHS_PINN*.5), int(EPOCHS_PINN*.8)], gamma=0.5)

print("Training PINN ...")
for ep in range(1, EPOCHS_PINN + 1):
    opt.zero_grad()
    t_c = torch.rand(500, 1, device=DEVICE) * T_EVAL[1]
    loss, ld, lp = pinn.total_loss(t_tr, x_tr, t_c)
    loss.backward(); opt.step(); sch.step()
    if ep % 4000 == 0:
        print(f"  ep {ep:5d}  data={ld.item():.4e}  phys={lp.item():.4e}")

In [ ]:
EPOCHS_NN = 3_000

nn_model = TrajNN(t_scale=T_TRAIN[1]).to(DEVICE)
opt_nn   = torch.optim.Adam(nn_model.parameters(), lr=1e-3)
sch_nn   = torch.optim.lr_scheduler.MultiStepLR(
    opt_nn, milestones=[int(EPOCHS_NN*.5), int(EPOCHS_NN*.8)], gamma=0.5)

print("Training NN ...")
for ep in range(1, EPOCHS_NN + 1):
    opt_nn.zero_grad()
    loss_nn = ((nn_model(t_tr) - x_tr)**2).mean()
    loss_nn.backward(); opt_nn.step(); sch_nn.step()
print(f"  final loss={loss_nn.item():.4e}")

In [ ]:
with torch.no_grad():
    x_pinn = pinn(t_ev).cpu().numpy().squeeze()
    x_nn   = nn_model(t_ev).cpu().numpy().squeeze()

split = T_TRAIN[1]
in_m, out_m = t_eval <= split, t_eval > split

metrics = {
    "PINN": {"interp": float(np.sqrt(np.mean((x_pinn[in_m]  - x_true[in_m])**2))),
             "extrap": float(np.sqrt(np.mean((x_pinn[out_m] - x_true[out_m])**2)))},
    "NN":   {"interp": float(np.sqrt(np.mean((x_nn[in_m]    - x_true[in_m])**2))),
             "extrap": float(np.sqrt(np.mean((x_nn[out_m]   - x_true[out_m])**2)))},
}
print(f"{'Model':<8} {'Interp RMSE':>13} {'Extrap RMSE':>13}")
print("-" * 36)
for name, m in metrics.items():
    print(f"{name:<8} {m['interp']:>13.4f} {m['extrap']:>13.4f}")
print(f"\nPINN extrap improvement: {metrics['NN']['extrap']/(metrics['PINN']['extrap']+1e-12):.1f}×")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
fig.suptitle(f"PINN vs NN — m={M} kg, c={C} N·s/m, k={K} N/m\n"
             f"{N_TRAIN} obs from [0, {T_TRAIN[1]} s]  |  shaded = extrapolation",
             fontsize=12, fontweight="bold")

ax1.plot(t_eval, x_true, color=COLORS["True"], lw=1, ls="--", label="Ground truth", zorder=5)
ax1.plot(t_eval, x_pinn, color=COLORS["PINN"], lw=1.8, label=f"PINN  (extrap={metrics['PINN']['extrap']:.4f})")
ax1.plot(t_eval, x_nn,   color=COLORS["NN"],   lw=1.8, label=f"NN    (extrap={metrics['NN']['extrap']:.4f})")
ax1.scatter(t_train, x_train, s=60, color=COLORS["True"], zorder=10,
            label="training obs", edgecolors="white", linewidths=0.5)
ax1.axvline(split, color="gray", ls=":", lw=1.4, label="train end")
ax1.axvspan(split, t_eval[-1], alpha=0.07, color="gray")
ax1.set_ylabel("x(t)", fontsize=11); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax1.set_title("Trajectory rollout", fontsize=11, fontweight="bold")

for name, pred, c in [("PINN", x_pinn, COLORS["PINN"]), ("NN", x_nn, COLORS["NN"])]:
    ax2.semilogy(t_eval, np.abs(pred - x_true) + 1e-6, color=c, lw=1.6, label=name)
ax2.axvline(split, color="gray", ls=":", lw=1.4)
ax2.axvspan(split, t_eval[-1], alpha=0.07, color="gray")
ax2.set_ylabel("|error|  (log scale)", fontsize=11); ax2.set_xlabel("Time (s)", fontsize=11)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "pinn_vs_nn.png"), dpi=150, bbox_inches="tight")
plt.show()